## 3.14 Libraries

Aaditya Prem (rex0dex). The core rule of this lesson: don't rewrite code that already exists and has been tested. Find a library, read its API, import it, and call its procedures.

| Hack | What I did |
|---|---|
| Popcorn 1 | Read the API for `random.randint` and `math.ceil`, changed the jersey range to 1-55 and the court to 196 sq ft |
| Popcorn 2 | Put my own soccer roster in Flask and added a `/record` route |
| Homework | Season stats report for my soccer season using `json`, `statistics`, and `datetime` |

Every cell only imports standard library modules, except Popcorn 2, which uses `flask` (the lesson's submission rules allow it in the code runner).


## Popcorn Hack 1: Read the API, Then Use It

**What I changed:** the jersey range from `randint(1, 99)` to `randint(1, 55)`, and the court from 144 sq ft to **196 sq ft**. I also print only the first line of each docstring so the API summary is short, and I seed `random` so the "random" picks are the same every run and I can check them.

**What the API says:**

* `random.randint(a, b)` returns a random integer from `a` to `b`, **including both** ends. So `randint(1, 55)` can return 1 or 55, never 0 or 56.
* `math.ceil(x)` returns the smallest integer that is `>= x`, so it always rounds **up**.
* `math.sqrt(x)` returns a float, so the side prints as `14.0`, not `14`.

**Predictions before running:**

* Jersey number: some whole number from 1 to 55 (I can't predict which without running the seed, but it must be in that range).
* Court side: `math.sqrt(196)` = **14.0** feet.
* Average: (24 + 18 + 31 + 27 + 22) / 5 = 122 / 5 = **24.4**, and `math.ceil(24.4)` = **25**.


In [ ]:
# CODE_RUNNER: 3.14 Popcorn 1 - jersey range 1-55 and a 196 sq ft court
import math
import random

random.seed(314)   # same "random" results every run so the output can be checked

# Read the API before calling a procedure (first line of each docstring)
print("random.randint:", random.randint.__doc__.strip().splitlines()[0])
print("math.ceil:", math.ceil.__doc__.strip().splitlines()[0])

points_per_game = [24, 18, 31, 27, 22]

# random.choice picks one item out of a list
print("Random game to review:", random.choice(points_per_game), "points")

# random.randint(a, b) returns a whole number from a to b, including both
jersey = random.randint(1, 55)
print("Random jersey number:", jersey)

# math.sqrt returns the square root, so a 196 square foot court is this wide
side = math.sqrt(196)
print("Square court side:", side, "feet")

# math.ceil rounds up to the next whole number
average = sum(points_per_game) / len(points_per_game)
rounded_up = math.ceil(average)
print("Points per game:", average, "-> rounded up:", rounded_up)


**Output:**

    random.randint: Return random integer in range [a, b], including both end points.
    math.ceil: Return the ceiling of x as an Integral.
    Random game to review: 18 points
    Random jersey number: 30
    Square court side: 14.0 feet
    Points per game: 24.4 -> rounded up: 25

Every prediction held: the jersey (30) is between 1 and 55, `math.sqrt(196)` is `14.0` (a float, because `sqrt` always returns a float), and `math.ceil(24.4)` rounds **up** to `25`. Because of `random.seed(314)`, the game to review (18) and the jersey (30) are the same every time I run the cell; delete the seed line and they change each run but the jersey stays in 1-55.

I never saw how `randint` or `ceil` work inside. The one-line API was enough to use them correctly. That is abstraction.


## Popcorn Hack 2: Write a Flask Route

**What I changed:** my own soccer roster in `/roster`, my own match in `/score`, and a new route **`/record`** that returns the team's wins, losses, and draws. It is defined above `app.test_client()`, like the lesson says.

**Prediction:** each `client.get(...)` visits a route and `.get_json()` turns the JSON back into a dictionary. One thing to expect: Flask's `jsonify` **sorts dictionary keys alphabetically** by default, so `/score` comes back as `away, half, home`, not in the order I typed them.


In [ ]:
# CODE_RUNNER: 3.14 Popcorn 2 - my roster and a new /record route
from flask import Flask, jsonify

app = Flask(__name__)

# A route is a procedure with a web address attached to it
@app.route("/score")
def score():
    return jsonify({"home": 3, "away": 1, "half": 2})

@app.route("/roster")
def roster():
    return jsonify({"players": ["Aaditya", "Rohan", "Ethan", "Kai"]})

# My route: the season record so far
@app.route("/record")
def record():
    return jsonify({"wins": 4, "losses": 1, "draws": 1})

# test_client visits the addresses without needing a browser
client = app.test_client()

print("/score says:", client.get("/score").get_json())
print("/roster says:", client.get("/roster").get_json())
print("/record says:", client.get("/record").get_json())


**Output:**

    /score says: {'away': 1, 'half': 2, 'home': 3}
    /roster says: {'players': ['Aaditya', 'Rohan', 'Ethan', 'Kai']}
    /record says: {'draws': 1, 'losses': 1, 'wins': 4}

All three routes answer. The keys come back sorted (`draws, losses, wins`), but the list inside `/roster` keeps its order, because sorting only applies to dictionary keys. My extra route is **`/record`**. `Flask` is an external library, so a real project would need `Flask` in its `requirements.txt` (see the homework).


## Homework Hack: Season Stats Report

My sport is **soccer**, so goals replace points. My version:

1. **Uses 3 standard library modules:** `json` (prints the report), `statistics` (`mean` and `median`), and `datetime` (`date`, for days until my next match).
2. **Has `summarize_season`**, which returns total goals, average goals, the best game, and goals against each opponent.
3. **Documents its API** with a docstring giving the parameter and the return value, so someone could call it without reading the loop.
4. **Uses `datetime`** to count the days until my next match on **2026-10-21**.
5. **Writes a `requirements.txt`** for a real version using `pandas` and `Flask`, with a comment on why `json`, `statistics`, and `datetime` are not listed.

**Predictions (worked out by hand):**

* Goals: 2 + 0 + 3 + 1 + 2 = **8** total over 5 matches.
* Average: 8 / 5 = **1.6**. Median of [0, 1, 2, 2, 3] = **2**.
* Best game: **3 goals vs Westview** on 2026-09-26.
* By opponent: Westview played twice (2 + 3 = **5**), Rancho Bernardo **0**, Del Norte **1**, Poway **2**.
* Days until next match: 2026-10-21 minus 2026-10-07 = **14** (when run on October 7, 2026).


In [ ]:
# CODE_RUNNER: 3.14 Homework - Season Stats Report (soccer)
import json
import statistics
from datetime import date

games = [
    {"date": "2026-09-12", "opponent": "Westview", "goals": 2, "minutes": 70},
    {"date": "2026-09-19", "opponent": "Rancho Bernardo", "goals": 0, "minutes": 55},
    {"date": "2026-09-26", "opponent": "Westview", "goals": 3, "minutes": 80},
    {"date": "2026-09-30", "opponent": "Del Norte", "goals": 1, "minutes": 65},
    {"date": "2026-10-03", "opponent": "Poway", "goals": 2, "minutes": 75},
]

def summarize_season(season):
    """
    Summarize a soccer season.

    Parameters:
        season: list of dictionaries, one per match, each with
                "date" (str, YYYY-MM-DD), "opponent" (str),
                "goals" (int), and "minutes" (int)
    Returns:
        dictionary with
            total_goals: int, sum of goals in every match
            average_goals: float, mean goals per match, rounded to 1 decimal
            median_goals: middle goals value of the season
            best_game: dictionary with the date, opponent, and goals of the
                       highest scoring match (the first one if there is a tie)
            goals_by_opponent: dictionary of opponent -> total goals against them
    """
    goals = [game["goals"] for game in season]

    goals_by_opponent = {}
    for game in season:
        opponent = game["opponent"]
        goals_by_opponent[opponent] = goals_by_opponent.get(opponent, 0) + game["goals"]

    best = max(season, key=lambda game: game["goals"])

    return {
        "total_goals": sum(goals),
        "average_goals": round(statistics.mean(goals), 1),
        "median_goals": statistics.median(goals),
        "best_game": {"date": best["date"], "opponent": best["opponent"], "goals": best["goals"]},
        "goals_by_opponent": goals_by_opponent,
    }

report = summarize_season(games)

# Subtracting two dates returns a timedelta, and .days is the whole number of days
next_game = date(2026, 10, 21)
report["next_game"] = next_game.isoformat()
report["days_until_next_game"] = (next_game - date.today()).days

print(json.dumps(report, indent=2))

# requirements.txt lists only external packages that pip has to install.
requirements_txt = """# json, statistics, and datetime are part of Python's standard library.
# They come with Python, so pip never installs them and they are not listed.
pandas
Flask
"""
print()
print("requirements.txt:")
print(requirements_txt)


**Output (run on October 7, 2026):**

    {
      "total_goals": 8,
      "average_goals": 1.6,
      "median_goals": 2,
      "best_game": {
        "date": "2026-09-26",
        "opponent": "Westview",
        "goals": 3
      },
      "goals_by_opponent": {
        "Westview": 5,
        "Rancho Bernardo": 0,
        "Del Norte": 1,
        "Poway": 2
      },
      "next_game": "2026-10-21",
      "days_until_next_game": 14
    }

    requirements.txt:
    # json, statistics, and datetime are part of Python's standard library.
    # They come with Python, so pip never installs them and they are not listed.
    pandas
    Flask

Every value matches my predictions. `days_until_next_game` uses `date.today()`, so it is **14** on October 7 and goes down by one each day; after the match it turns negative.

**Why the library versions win:** `statistics.mean` and `statistics.median` replace two loops I would otherwise write and test myself, and `json.dumps(..., indent=2)` turns the nested dictionary into readable text (the same format a Flask scoreboard route would send) in one call. `datetime` handles month lengths for me: September has 30 days, but I never had to think about that.

**`requirements.txt`:** a real version of this app would use `pandas` for the season table and `Flask` for the scoreboard page, so those are the only two lines. `json`, `statistics`, and `datetime` ship with Python. If a teammate cloned the project and saw `ModuleNotFoundError: No module named 'flask'`, the fix is `pip install -r requirements.txt` inside an active `venv`.


## Submission notes

Pasted into the description box:

    Lesson: CSP 3.14 Libraries
    Popcorn 1: what I changed = jersey range 1-99 to 1-55 (seeded random), court 144 to 196 sq ft so side = 14.0
    Popcorn 2: my extra route = /record
    Homework: libraries used = json, statistics, datetime
    Homework: days until my next game = 14 (next match 2026-10-21, counted on 2026-10-07)
